> ### ▶ Running this notebook in Google Colab (recommended)> 1. Go to [colab.research.google.com](https://colab.research.google.com)> 2. Click **File → Upload notebook**> 3. Select this `.ipynb` file from your downloaded course ZIP> 4. Click **Runtime → Run all**>> Alternatively: upload the whole ZIP to **Google Drive**, then double-click any notebook — it opens in Colab automatically.>> _No installation required. All you need is a free Google account._

## 3.1 — Setup

In [ ]:
import sysIN_COLAB = 'google.colab' in sys.modulesif IN_COLAB:    import subprocess    subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', '--quiet'])import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerprint("✅ Ready")

---## 3.2 — The Assumptions BlockThe single biggest improvement Python brings to financial modelling: all assumptions in one place.In Excel, assumptions are often buried in cells across multiple tabs. Here they live in a single dictionary — change one value and the entire model updates.

In [ ]:
# ── ALL MODEL ASSUMPTIONS LIVE HERE ─────────────────────────────────────────assumptions = {    # Revenue    'base_revenue':       10_000,   # £000s, Year 1    'revenue_growth':     [0.12, 0.10, 0.09],  # YoY growth rates Y1→Y2, Y2→Y3    # Margins (as % of revenue)    'gross_margin':       0.62,     # 62%    'opex_pct_revenue':   0.28,     # 28%    'da_fixed':           500,      # £000s per year (fixed, not % of revenue)    # Tax    'tax_rate':           0.25,     # 25% corporation tax}years = ['Year 1', 'Year 2', 'Year 3']print("Assumptions loaded. Model covers:", years)

---## 3.3 — Build the P&L

In [ ]:
def build_pl(assumptions, years):    """Build a 3-year P&L from an assumptions dictionary."""    a = assumptions    pl = pd.DataFrame(index=years)    # Revenue    rev = [a['base_revenue']]    for g in a['revenue_growth'][1:]:        rev.append(rev[-1] * (1 + g))    pl['Revenue'] = rev    # Cost of Goods Sold & Gross Profit    pl['COGS']          = pl['Revenue'] * (1 - a['gross_margin'])    pl['Gross Profit']  = pl['Revenue'] - pl['COGS']    pl['Gross Margin %'] = pl['Gross Profit'] / pl['Revenue']    # Operating Expenses    pl['OpEx']          = pl['Revenue'] * a['opex_pct_revenue']    pl['EBITDA']        = pl['Gross Profit'] - pl['OpEx']    pl['EBITDA Margin %'] = pl['EBITDA'] / pl['Revenue']    # D&A and EBIT    pl['D&A']           = a['da_fixed']    pl['EBIT']          = pl['EBITDA'] - pl['D&A']    pl['EBIT Margin %'] = pl['EBIT'] / pl['Revenue']    # Tax and Net Income    pl['Tax']           = pl['EBIT'] * a['tax_rate']    pl['Net Income']    = pl['EBIT'] - pl['Tax']    pl['Net Margin %']  = pl['Net Income'] / pl['Revenue']    return pl.round(1)pl = build_pl(assumptions, years)pl

In [ ]:
# Format for presentation — like formatting an Excel tabledef format_pl(pl):    display = pl.copy()    pct_cols = [c for c in display.columns if '%' in c]    val_cols = [c for c in display.columns if '%' not in c]    for c in val_cols:        display[c] = display[c].apply(lambda x: f"£{x:,.0f}k")    for c in pct_cols:        display[c] = display[c].apply(lambda x: f"{x*100:.1f}%")    return displayprint("3-Year P&L Model (£000s)")print("=" * 55)format_pl(pl)

---## 3.4 — Sensitivity AnalysisIn Excel, sensitivity tables require the Data Table feature — awkward, slow, and limited to 2 variables.In Python, you loop over any number of variables in seconds.

In [ ]:
# Sensitivity: EBIT vs Revenue Growth Rate and Gross Margingrowth_rates  = [0.06, 0.08, 0.10, 0.12, 0.14]gross_margins = [0.55, 0.58, 0.62, 0.65, 0.68]# Build sensitivity table (Year 3 EBIT)sensitivity = pd.DataFrame(index=[f"{g*100:.0f}% growth" for g in growth_rates],                            columns=[f"{m*100:.0f}% GM" for m in gross_margins])for g in growth_rates:    for m in gross_margins:        custom = assumptions.copy()        custom['revenue_growth'] = [g, g, g]        custom['gross_margin'] = m        result = build_pl(custom, years)        sensitivity.loc[f"{g*100:.0f}% growth", f"{m*100:.0f}% GM"] = round(result.loc['Year 3','EBIT'], 0)print("Year 3 EBIT Sensitivity (£000s) — Revenue Growth vs Gross Margin")sensitivity

---## 3.5 — Scenario Analysis

In [ ]:
scenarios = {    'Bear': {**assumptions, 'revenue_growth': [0.05, 0.04, 0.04], 'gross_margin': 0.56, 'opex_pct_revenue': 0.31},    'Base': assumptions,    'Bull': {**assumptions, 'revenue_growth': [0.18, 0.15, 0.13], 'gross_margin': 0.67, 'opex_pct_revenue': 0.25},}scenario_summary = {}for name, assum in scenarios.items():    result = build_pl(assum, years)    scenario_summary[name] = result['EBIT']scenario_df = pd.DataFrame(scenario_summary).Tscenario_df.columns = yearsprint("EBIT by Scenario (£000s):")scenario_df

In [ ]:
# Chart: scenario comparisonfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Left: Revenue by scenarioax = axes[0]for name, assum in scenarios.items():    result = build_pl(assum, years)    ax.plot(years, result['Revenue'], marker='o', linewidth=2, label=name)ax.set_title('Revenue by Scenario (£000s)')ax.set_ylabel('£000s')ax.legend()ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))# Right: EBIT margin by scenarioax2 = axes[1]for name, assum in scenarios.items():    result = build_pl(assum, years)    ax2.plot(years, result['EBIT Margin %'] * 100, marker='s', linewidth=2, label=name)ax2.set_title('EBIT Margin % by Scenario')ax2.set_ylabel('EBIT Margin %')ax2.legend()ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))plt.tight_layout()plt.show()

---## 3.6 — Key Takeaways- A single assumptions dictionary replaces scattered hard-coded cells across Excel tabs- The `build_pl()` function makes the model reusable — swap in new assumptions and re-run instantly- Sensitivity tables in Python are a simple nested loop — no Excel Data Table wizardry required- Scenario analysis is clean and explicit: named dictionaries, one chart per metric- Every chart is reproducible — no manual reformatting each month---## Up Next: Module 4 — DCF ValuationWe'll build an end-to-end discounted cash flow model with a dynamic WACC, terminal value, and sensitivity on entry multiples.